# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RohanRamchandani/FlyRank-ML-Week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/FlyRank-ML-Week1/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/FlyRank-ML-Week1/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/FlyRank-ML-Week1/data/processed/model_predictions.csv
Wrote model results: /content/FlyRank-ML-Week1/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/FlyRank-ML-Week1/outputs/refresh_queue.csv
Wrote model report: /content/FlyRank-ML-Week1/outputs/mode

**Task type: Ranking / Scoring**

Using the framing skill's mapping table, my question is "which page should an editor review
first?" — that matches "Which ones first?" → **Ranking / scoring**, target = a priority score,
metric = precision@K.

Walking through the skill's four framing questions:
1. **What decision does this improve?** Which pages an editor should review first, given
   limited capacity — not "predict decline" in the abstract, but an actual prioritization decision.
2. **Who acts, and what do they do?** A content/SEO editor with limited review time acts on the
   ranked queue by choosing which pages to open and fix first.
3. **What does a wrong answer cost?** A false positive wastes editor hours on a healthy page;
   a false negative lets a real decline go unnoticed and unaddressed.
4. **Why does data/ML help?** Because decline depends on several signals interacting messily
   (staleness, position, CTR, volume) in ways that shift by content type — too tangled for one
   fixed if-statement to capture well.

It is not clustering (no unlabeled groups needed), not plain signal analysis (I need an
actionable ordered output, not just correlations), and not classification alone (a single
yes/no per page isn't the deliverable — the ranking is).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Target / proxy:** `is_declining_label = (trend_direction == "down")`

Per the framing skill's rule — "the target must be observed, not defined by a rule, or your
model just learns the rule" — I have to be honest that this target is **defined**, not purely
observed: `trend_direction` is a bucket computed from `trend_pct`, which is itself derived
from the data, not an independently observed future outcome.

**Data-skill label trap:** because `is_declining_label` is derived from `trend_direction`,
which is computed from `trend_pct`, both `trend_direction` and `trend_pct` must NEVER be used
as model features — only as the source of the label itself. Using either as a feature would be
leakage: the model would just read the answer back off a reworded copy of itself.

A stronger, genuinely observed version for later work:
`features from the prior 90 days -> decline over the next 30 days` — an outcome measured in a
later time window, which is what the framing skill recommends over a same-window derived label.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Label built ONLY from trend_direction -- never used as a feature (label trap, per data skill)
y = (df["trend_direction"] == "down").astype(int)

def precision_at_k(scores, labels, k):
    order = scores.sort_values(ascending=False).index
    topk = labels.loc[order[:k]]
    return topk.mean()

# reference score using a safe, non-leaky signal
naive_score = df["impressions_90d"]
print(f"Naive impressions-only Precision@50: {precision_at_k(naive_score, y, 50):.3f}")
print("Reference range: starter hand-written rule = 0.240, starter random forest = 0.740")

Naive impressions-only Precision@50: 0.420
Reference range: starter hand-written rule = 0.240, starter random forest = 0.740


**Success metric: Precision@50**

Matches the task-type table's guidance for ranking/scoring problems: "Which ones first?" →
precision@K. This fits the real constraint — an editor can only act on a limited number of
pages, so getting the top of the list right matters more than accuracy across all 30,000 rows.

Per the framing skill's rule "name the metric before training," I'm defending a concrete
threshold now, computed from the starter pipeline's already-verified baseline: 0.240
(hand-written rule). Any scoring method must clearly beat that number to justify using
data/ML over a fixed rule.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
cols = ["content_id", "client_id", "impressions_90d", "sessions_90d",
        "content_age_days", "days_since_last_update", "word_count",
        "avg_position", "ctr", "engagement_rate"]

lane2_slice = df[cols].copy()

# avg_position == 0 means "no data" per the data dictionary -- flag it, don't treat as rank 0
lane2_slice["has_position_data"] = lane2_slice["avg_position"] > 0
no_position = (lane2_slice["avg_position"] == 0).sum()

print(f"Rows: {len(lane2_slice)}  |  Unique content_id: {lane2_slice['content_id'].nunique()}")
print(f"Rows with avg_position == 0 (no data, not rank zero): {no_position}")
lane2_slice.head(5)

Rows: 30000  |  Unique content_id: 30000
Rows with avg_position == 0 (no data, not rank zero): 1205


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,days_since_last_update,word_count,avg_position,ctr,engagement_rate,has_position_data
0,content_304f48230142,client_f369cb89fc,3803,17,187,20,3221.0,10.6,0.76,5.88,True
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,25,2481.0,20.3,0.05,0.00,True
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,20,3515.0,36.5,0.09,0.00,True
3,content_331d6c4de07b,client_19581e27de,11751,78,463,22,NaN,6.2,0.49,1.28,True
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,14,2803.0,44.0,0.13,0.00,True


**Unit of analysis: one row = one content page (`content_id`), summarized over its trailing
90-day window, belonging to one pseudonymized client.**

This is the right grain because the decision — "should someone review this page?" — is made
per page, not per client or per daily event.

Data-skill gotchas accounted for below:
- `ctr` and `engagement_rate` are already ×100 percentages (`ctr = 0.76` means 0.76%, not 76%)
  — not re-scaled or misread here.
- `avg_position = 0` means "no position data," not literally rank zero — flagged with a
  `has_position_data` column rather than treated as a real position.
- `content_id` / `client_id` are pseudonymous join keys only — shown for identification,
  never used as model features.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json
res = json.load(open("outputs/model_results.json"))

base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]

print(f"Hand-written rule Precision@50: {base:.3f}")
print(f"Random forest Precision@50:     {rf:.3f}")
print(f"Improvement: {rf/base:.1f}x")

Hand-written rule Precision@50: 0.240
Random forest Precision@50:     0.740
Improvement: 3.1x


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.